# Exercise 1 — Bootstrap Hypothesis Testing on Neural Spike Trains

**Neural Data Science (WP7)** · Week 2 (released 20.04, presented 23.04)

## The question

A neuron in primary visual cortex (V1) fires in response to visual stimuli. Look at the inter-spike intervals (ISIs) — the times between consecutive spikes.

**Are those ISIs more regular than you'd expect by chance?**

If the neuron fired completely at random (Poisson process), ISIs would follow an exponential distribution. Real neurons often don't — they have refractory periods, oscillatory drive, bursts. The bootstrap is one way to test whether the pattern you see could plausibly come from "random".

## Learning objectives

- Build a null distribution by resampling / shuffling
- Frame a null hypothesis in terms of what you permute
- Compute a p-value without parametric assumptions
- Decide what "surprising" means in practice

## Prerequisites & resources

**Lectures:** Wiktor's Lec 01a + 01b (Introduction & nonparametric testing).

**Reading (pick what helps):**

- Murphy — *Machine Learning: A Probabilistic Approach*, §4.1–4.3 (intro to probability) and §6.5 (bootstrap).
- MacKay — *Information Theory, Inference and Learning Algorithms*, Ch. 29 (Monte Carlo basics).
- Efron & Tibshirani (1986), [*Bootstrap methods for standard errors, confidence intervals, and other measures of statistical accuracy*](https://projecteuclid.org/journals/statistical-science/volume-1/issue-1/Bootstrap-Methods-for-Standard-Errors-Confidence-Intervals-and-Other-Measures/10.1214/ss/1177013815.full) — the classic first paper on the bootstrap.
- [SciPy `bootstrap` API](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.bootstrap.html) (reference; don't use it directly for this exercise — write your own).

**Data:** [CRCNS PVC-8](https://crcns.org/data-sets/vc/pvc-8) — extracellular recordings from V1 of anaesthetised macaques responding to natural image patches.

## How to work through this

Each section has a short prompt and an empty code cell. **Explore freely.** There's a "peek if stuck" collapsible at the end of each section with a skeleton — use it only after you've tried on your own. No autograder, no required signature for your code — plot things, print things, compare approaches.

Estimated time: **2–3 hours**.


## Setup


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.io
import os

%matplotlib inline
rng = np.random.default_rng(42)  # for reproducibility in bootstrap loops


## Load the data

Single-unit recordings from V1. The `.mat` file contains a 4-D array `resp_train` with shape `(n_cells, n_images, n_trials, n_tp)` — binary spike trains (0 or 1 per millisecond time bin).


In [ ]:
data_path = '../../data/crcns_pvc8/2.mat'
mat = scipy.io.loadmat(data_path)
R = mat['resp_train']
n_cells, n_images, n_trials, n_tp = R.shape
print(f'Data shape: {R.shape}')
print(f'  {n_cells} neurons, {n_images} images, {n_trials} trials, {n_tp} time bins (ms)')


---
## Section 1 — Explore the data

Before doing any statistics, *look* at the data.

Pick a neuron and visualise its spike trains. What do you notice? Is the firing rate constant over the response window? Does it vary across trials? Across images?

**Questions to keep in mind:**
- What's the *time axis* in one trial? (It's in the README for the dataset.)
- Are responses stimulus-locked? Where in the time axis do spikes concentrate?
- How much variability is there across repeated trials of the same image?


In [ ]:
# Your exploration here — plot rasters, firing rates, whatever helps you understand the data.



<details>
<summary>Peek if stuck — a skeleton for a single-neuron raster</summary>

```python
neuron_idx = 10
image_idx = 5
fig, ax = plt.subplots(figsize=(10, 4))
ax.imshow(R[neuron_idx, image_idx], cmap='binary', aspect='auto', interpolation='nearest')
ax.set_xlabel('Time (ms)'); ax.set_ylabel('Trial')
```

To get *all* trials for one neuron across all images as one 2-D array: `R[neuron_idx].reshape(-1, n_tp)`.
</details>


---
## Section 2 — Inter-spike intervals (ISIs)

Pick one neuron and compute its ISIs across all images and trials. An ISI is the time between two consecutive spikes on the same trial.

Plot the ISI distribution as a histogram.

**Think about:**
- What shape do you expect if spikes were generated by a Poisson process? (Hint: this is covered in Wiktor's lecture.)
- Does the observed distribution match? Where does it differ?
- What single number would you use to summarise the distribution? Mean? Median? Mode? CV? — there's no one right answer; pick what feels informative.

Your *observed test statistic* is whatever summary you choose of these real ISIs. Write it down — you'll compare it against a null in Section 3.


In [ ]:
# Compute ISIs for one neuron; plot the distribution; pick a test statistic.



<details>
<summary>Peek if stuck — extracting ISIs from a binary raster</summary>

For each trial (each row of a 2-D binary array), `np.where(row == 1)[0]` gives spike time indices. `np.diff()` of those indices gives ISIs in units of time bins. Loop over trials, concatenate.

Be careful with trials that have fewer than 2 spikes — they contribute no ISIs.
</details>


---
## Section 3 — Build a null distribution

You need a null: *what would the ISI distribution look like if the temporal structure you saw were absent?*

One natural choice is **time-bin shuffling**: for each trial, permute the 0/1 values across time. This preserves the *count* of spikes per trial but destroys any temporal correlation.

Run the shuffle 1000 times. On each iteration, re-compute your test statistic. Collect them into a null distribution.

**Think about:**
- What exactly is the null hypothesis you're testing? Write it down in words.
- Would a different shuffling scheme (e.g. shuffling trials, or shifting all spikes by a random offset) answer a *different* question? Which scheme is right for *your* question?
- 1000 is a common default. Is that enough for the p-value you care about? (The smallest p-value you can detect is ≈ 1/n_bootstrap.)


In [ ]:
# Build the null distribution: ≥ 1000 iterations, each shuffling + recomputing your statistic.



<details>
<summary>Peek if stuck — per-trial time-bin shuffle</summary>

`np.random.permutation(row)` permutes the elements of `row`. Apply it independently to each trial inside your loop. Then re-run your ISI computation on the shuffled array.

1000 iterations of a 500-trial shuffle takes ~30 seconds. A print every 250 iterations is enough to see progress.
</details>


---
## Section 4 — Draw a conclusion

Compare your observed test statistic to the null distribution.

- Plot the null distribution and mark the observed value with a vertical line.
- Compute a p-value. Decide on one-sided or two-sided and justify your choice.
- Report the result in one sentence a non-statistician would understand.

**Think about:**
- Is the p-value itself informative, or is the *effect size* (how far the observed stat lies from the null centre) more useful?
- If the p-value is tiny (< 0.001), what does that tell you? What does it *not* tell you?
- What would change if you picked a different neuron? Re-run the whole pipeline on 2–3 other neurons and see whether the conclusion holds.


In [ ]:
# Compute p-value, plot null vs observed, write a one-sentence conclusion.



<details>
<summary>Peek if stuck — two-sided p-value formula</summary>

```python
null_center = np.mean(null_stats)
p_value = np.mean(np.abs(null_stats - null_center) >= np.abs(observed_stat - null_center))
```

This asks: "How often does the null produce a value at least as far from its centre as my observation?"
</details>


---
## Reflection

Before you submit, answer these in a markdown cell below:

1. What was your test statistic, and why did you pick it?
2. What does your null say, in words? What biological assumption are you rejecting if it fails?
3. Did the conclusion hold across the neurons you tried? Where did it fail?
4. If a reviewer asked "could this just be a type-I error because you ran many tests?", how would you defend your analysis?

You're done when you could show this notebook to someone outside neuroscience and they'd follow both the logic and the conclusion.
